# Setup SageMaker Session

In [11]:
import boto3
import sagemaker
from sagemaker import get_execution_role
from sagemaker.inputs import TrainingInput
from sagemaker.estimator import Estimator

session = sagemaker.Session()
role    = get_execution_role()
region  = boto3.Session().region_name

print(f"Region : {region}")
print(f"Role   : {role}")
print("✅ Session ready")

Region : us-east-1
Role   : arn:aws:iam::504649076991:role/Ali-SageMakerExecutionRole
✅ Session ready


# Set Your S3 Bucket Name

In [12]:
# Adding my bucket name
BUCKET_NAME = "ali-car-price-new"  

PREFIX      = "car-price"

s3_train  = f"s3://{BUCKET_NAME}/{PREFIX}/train/"
s3_test   = f"s3://{BUCKET_NAME}/{PREFIX}/test/"
s3_output = f"s3://{BUCKET_NAME}/{PREFIX}/output/"

print("Train  :", s3_train)
print("Test   :", s3_test)
print("Output :", s3_output)

Train  : s3://ali-car-price-new/car-price/train/
Test   : s3://ali-car-price-new/car-price/test/
Output : s3://ali-car-price-new/car-price/output/


# Verify Files Exist in S3

In [13]:
s3 = boto3.client('s3')

def check_s3(bucket, prefix):
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    files = [obj['Key'] for obj in resp.get('Contents', [])]
    return files

train_files = check_s3(BUCKET_NAME, f"{PREFIX}/train/")
test_files  = check_s3(BUCKET_NAME, f"{PREFIX}/test/")

print("Train files:", train_files)
print("Test  files:", test_files)

assert len(train_files) > 0, "❌ train.csv not found in S3!"
assert len(test_files)  > 0, "❌ test.csv not found in S3!"
print("\n✅ Both files confirmed in S3!")

Train files: ['car-price/train/', 'car-price/train/train.csv']
Test  files: ['car-price/test/', 'car-price/test/test.csv']

✅ Both files confirmed in S3!


# Get XGBoost Container

In [14]:
container = sagemaker.image_uris.retrieve(
    framework = "xgboost",
    region    = region,
    version   = "1.7-1"
)
print("Container:", container)
print("✅ XGBoost container ready")

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


Container: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1
✅ XGBoost container ready


# Create the Estimator

In [15]:
estimator = Estimator(
    image_uri         = container,
    role              = role,
    instance_count    = 1,
    instance_type     = "ml.m5.large",
    output_path       = s3_output,
    sagemaker_session = session,
    base_job_name     = "car-price-xgb"
)

print("✅ Estimator created")

✅ Estimator created


# Set Hyperparameters

In [18]:
estimator.set_hyperparameters(
    objective             = "reg:squarederror",  # regression task
    eval_metric           = "rmse",
    num_round             = 150,
    max_depth             = 6,
    eta                   = 0.1,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    min_child_weight      = 5,
    early_stopping_rounds = 15
)

print("✅ Hyperparameters set")

✅ Hyperparameters set


# Define Training Inputs

In [19]:
train_input = TrainingInput(s3_train, content_type="text/csv")
test_input  = TrainingInput(s3_test,  content_type="text/csv")

print("✅ Inputs defined")

✅ Inputs defined


# Start Training

In [20]:
# This will take about 3-8 minutes
# You will see RMSE printed every round below

estimator.fit(
    inputs = {"train": train_input, "validation": test_input},
    logs   = True
)

print("\n✅ TRAINING COMPLETE!")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: car-price-xgb-2026-03-02-20-17-56-662


2026-03-02 20:17:57 Starting - Starting the training job...
2026-03-02 20:18:13 Starting - Preparing the instances for training...
2026-03-02 20:18:37 Downloading - Downloading input data...
2026-03-02 20:19:28 Downloading - Downloading the training image......./miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[2026-03-02 20:20:35.853 ip-10-0-200-107.ec2.internal:7 INFO utils.py:28] RULE_JOB_STOP_SIGNAL_FILENAME: None
[2026-03-02 20:20:35.942 ip-10-0-200-107.ec2.internal:7 INFO profiler_config_parser.py:111] User has disabled profiler.
[2026-03-02:20:20:36:INFO] Imported framework sagemaker_xgboost_container.training
[2026-03-02:20:20:36:INFO] Failed to parse hyperparameter eval_metric value rmse to 

# View Results

In [23]:
job_name = estimator.latest_training_job.name
print("Job name     :", job_name)
print("Model saved  :", estimator.model_data)

sm = boto3.client('sagemaker')
desc = sm.describe_training_job(TrainingJobName=job_name)

print("Status       :", desc['TrainingJobStatus'])
print("Duration     :", desc.get('TrainingTimeInSeconds'), "seconds")
print("Billable time:", desc.get('BillableTimeInSeconds'), "seconds")

print("\nFinal Metrics:")
for m in desc.get('FinalMetricDataList', []):
    print(f"  {m['MetricName']:30s} = {m['Value']:.4f}")

Job name     : car-price-xgb-2026-03-02-20-17-56-662
Model saved  : s3://ali-car-price-new/car-price/output/car-price-xgb-2026-03-02-20-17-56-662/output/model.tar.gz
Status       : Completed
Duration     : 140 seconds
Billable time: 140 seconds

Final Metrics:
  train:rmse                     = 5060.1523
  validation:rmse                = 7889.9863


In [24]:
import pandas as pd
import numpy as np
import xgboost as xgb
import tarfile
import os
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Step 1: Download model from S3
s3 = boto3.client('s3')

# Extract bucket and key from model_data path
model_path = xgb.model_data  # s3://bucket/prefix/output/.../model.tar.gz
path_parts = model_path.replace('s3://', '').split('/')
model_bucket = path_parts[0]
model_key = '/'.join(path_parts[1:])

s3.download_file(model_bucket, model_key, 'model.tar.gz')
print("✅ Model downloaded!")

# Step 2: Extract model
with tarfile.open('model.tar.gz', 'r:gz') as tar:
    tar.extractall('.')
print("✅ Model extracted!")

# Step 3: Load model
booster = xgb.Booster()
booster.load_model('xgboost-model')
print("✅ Model loaded!")

# Step 4: Load test data
test_data = pd.read_csv('test.csv', header=None)
y_test = test_data.iloc[:, 0].values
X_test = test_data.iloc[:, 1:].values

dtest = xgb.DMatrix(X_test)

# Step 5: Predict
predictions = booster.predict(dtest)

# Step 6: Calculate ALL metrics
mae  = mean_absolute_error(y_test, predictions)
mse  = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, predictions)
mape = np.mean(np.abs((y_test - predictions) / y_test)) * 100

print("=" * 45)
print("📊        MODEL EVALUATION METRICS        ")
print("=" * 45)
print(f"  ✅ R² Score  (R-Squared)   : {r2:.4f}")
print(f"  ✅ MAE  (Mean Abs Error)   : {mae:.4f}")
print(f"  ✅ RMSE (Root Mean Sq Err) : {rmse:.4f}")
print(f"  ✅ MSE  (Mean Sq Error)    : {mse:.4f}")
print(f"  ✅ MAPE (Mean Abs % Err)   : {mape:.2f}%")
print("=" * 45)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│    1 import pandas as pd                                                                         │
│    2 import numpy as np                                                                          │
│ ❱  3 import xgboost as xgb                                                                       │
│    4 import tarfile                                                                              │
│    5 import os                                                                                   │
│    6 from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error               │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ModuleNotFoundError: No module named 'xgboost'